In [ ]:
# Lecture 22 — The Shuffle: trigger one, read the metrics in the Spark UI
# Spark 101 · Module 7 · Lecture 22
#
# Import this file into a Databricks workspace as a notebook.
# Databricks recognizes `# COMMAND ----------` as a cell boundary on import.
# In VS Code / PyCharm this stays a plain Python file — same source, two homes.
#
# BIG-DATA DEMO: run on a cluster large enough that the shuffle numbers are real.
# Warn about DBU cost on screen and keep auto-termination on.

In [ ]:
# Partition counts, serverless-safe.
# df.rdd.getNumPartitions() uses the RDD API, which is BLOCKED on serverless
# compute. We count non-empty partitions via spark_partition_id() instead.
from pyspark.sql.functions import spark_partition_id

def part_count(df):
    return df.select(spark_partition_id()).distinct().count()

In [ ]:
print("Spark version:", spark.version)
# Databricks gives you `spark` for free — no SparkSession.builder needed.

In [ ]:
# --- A dataset big enough for the shuffle to show up in the UI -------------
from pyspark.sql.functions import col, count, sum as _sum

# ~20M rows with a low-cardinality key to group on.
df = spark.range(0, 20_000_000).withColumn("key", (col("id") % 1000))
print("partitions:", part_count(df))

In [ ]:
# --- A WIDE op: groupBy triggers a shuffle --------------------------------
# Run this, then open the Spark UI -> Stages. Look for two stages with an
# Exchange between them, and the "Shuffle Write" / "Shuffle Read" columns.
grouped = df.groupBy("key").agg(count("*").alias("n"), _sum("id").alias("total"))
grouped.count()   # action — forces the shuffle to run

In [ ]:
# --- See the shuffle in the plan: the "Exchange" node -----------------------
grouped.explain()
# `Exchange hashpartitioning(key, ...)` IS the shuffle. One per wide op.

In [ ]:
# --- Filter EARLY: shrink the data before the shuffle ----------------------
# Same answer, but far less data crosses the network.
narrow_first = (
    df.filter(col("key") < 100)        # narrow, free — cut rows first
      .groupBy("key")                  # wide — now shuffles ~10% of the data
      .agg(count("*").alias("n"))
)
narrow_first.count()
narrow_first.explain()
# Compare the shuffle bytes in the UI vs the unfiltered groupBy above.

In [ ]:
# --- Count the shuffles: each wide op adds an Exchange ---------------------
multi = (
    df.groupBy("key").agg(_sum("id").alias("total"))  # shuffle 1
      .orderBy(col("total").desc())                    # shuffle 2
      .distinct()                                       # shuffle 3
)
multi.explain()
# Count the `Exchange` nodes — three wide ops, three shuffles.
# Every one you add is another disk -> network -> disk round trip.

In [ ]:
# A join is a shuffle too — both sides regroup by key.
dim = spark.range(0, 1000).withColumnRenamed("id", "key")
df.join(dim, "key").explain()     # Exchange on both sides (SortMergeJoin)

In [ ]:
# distinct() shuffles — identical rows must meet on one Worker.
df.select("key").distinct().explain()

In [ ]:
# Control shuffle width: fewer shuffle partitions for a small result.
spark.conf.set("spark.sql.shuffle.partitions", 16)
df.groupBy("key").count().explain()

In [ ]:
# Recap:
#   - A shuffle = write shuffle files -> transfer over network -> read + regroup.
#   - Triggered by every wide op: groupBy, join, distinct, orderBy, repartition.
#   - Costs disk + network + serialization, and creates a stage boundary.
#   - Minimize: filter early, select fewer columns, avoid needless wide ops.
# Next lecture: caching — stop recomputing (and re-shuffling) the same data.